# Pneumonia Detection from Chest X-rays

This notebook demonstrates an end-to-end supervised learning pipeline (CNN) for detecting pneumonia from chest X-ray images. It includes data download instructions, preprocessing, model training (TensorFlow/Keras), evaluation, visualization, and ethical reflection tied to SDG 3 (Good Health and Well-being).

## 1. Setup and Dependencies

Install packages and set up dataset. If using Kaggle, place kaggle.json in `~/.kaggle/` or follow Kaggle CLI setup. Alternatively, use the smaller 'chest_xray' dataset hosted on public mirrors.

In [1]:
%pip install -q tensorflow scikit-learn matplotlib seaborn opencv-python kaggle

# Standard imports
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure TensorFlow is available in this kernel; attempt to import and install only if needed.
try:
    import tensorflow as tf
except Exception as e:
    print("TensorFlow import failed, attempting to install and retry:", e)
    # Install TensorFlow into the current kernel/environment then import dynamically
    %pip install -q tensorflow
    import importlib
    tf = importlib.import_module('tensorflow')

# Use the imported `tf` module to access Keras objects.
keras = tf.keras
layers = tf.keras.layers

from sklearn.metrics import classification_report, confusion_matrix

print('TF', tf.__version__)
print('Python', sys.version)
print('CWD', os.getcwd())

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\user\\OneDrive\\Desktop\\SDG 3 — Pneumonia Detection from Chest X-rays\\.venv\\Lib\\site-packages\\tensorflow\\include\\external\\envoy_api\\envoy\\extensions\\load_balancing_policies\\client_side_weighted_round_robin\\v3\\client_side_weighted_round_robin.upb_minitable.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



Note: you may need to restart the kernel to use updated packages.
TensorFlow import failed, attempting to install and retry: No module named 'tensorflow.python'
TensorFlow import failed, attempting to install and retry: No module named 'tensorflow.python'


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\user\\OneDrive\\Desktop\\SDG 3 — Pneumonia Detection from Chest X-rays\\.venv\\Lib\\site-packages\\tensorflow\\include\\external\\envoy_api\\envoy\\extensions\\load_balancing_policies\\client_side_weighted_round_robin\\v3\\client_side_weighted_round_robin.upb_minitable.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'tensorflow.python'

## 2. Download dataset

This notebook expects the 'chest_xray' dataset arranged as: `chest_xray/train/PNEUMONIA`, `chest_xray/train/NORMAL`, and similarly for `val` and `test`. If you have Kaggle credentials, you can download the dataset from 'paultimothymooney/chest-xray-pneumonia'. Otherwise, download from a public mirror and extract into the notebook directory.

In [ ]:
# Helper: check for dataset in workspace
DATA_DIR = 'chest_xray'
if not os.path.exists(DATA_DIR):
    print('Dataset folder not found. Please download and extract the Chest X-Ray Pneumonia dataset into', DATA_DIR)
else:
    print('Found', DATA_DIR)
    for root, dirs, files in os.walk(DATA_DIR):
        print(root, '->', len(files), 'files')
        break

## 3. Preprocessing and Data Pipeline

We'll use Keras ImageDataGenerator for on-the-fly augmentation and resizing to 150x150. This keeps the notebook lightweight for training demonstrations.

In [ ]:
%pip install -q tensorflow

# Use the already-imported TensorFlow/Keras objects (defined in an earlier cell as `tf` / `keras`)
# to avoid editor/linter "could not be resolved" warnings. This reuses the `keras` variable
# set in the notebook (see cell 2 where `keras = tf.keras`).
ImageDataGenerator = keras.preprocessing.image.ImageDataGenerator

IMG_SIZE = (150, 150)
train_dir = os.path.join(DATA_DIR, 'train')
val_dir = os.path.join(DATA_DIR, 'val')
test_dir = os.path.join(DATA_DIR, 'test')

train_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, rotation_range=10, zoom_range=0.1)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=32, class_mode='binary')
val_gen = test_datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=32, class_mode='binary')
test_gen = test_datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=32, class_mode='binary', shuffle=False)

print('Classes:', train_gen.class_indices)

## 4. Model: small CNN

Define a compact CNN suitable for demonstration. For production, consider transfer learning (e.g., EfficientNet, ResNet).

In [ ]:
def make_model(input_shape=(*IMG_SIZE,3)):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = make_model()
model.summary()

## 5. Train (fast demo)

Train for a few epochs for demonstration. Increase epochs for better performance.

In [ ]:
EPOCHS = 5
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)

plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.show()

## 6. Evaluation

Compute confusion matrix and classification report on the test set.

In [ ]:
preds = model.predict(test_gen)
y_pred = (preds.ravel() > 0.5).astype(int)
y_true = test_gen.classes
print(classification_report(y_true, y_pred, target_names=list(test_gen.class_indices.keys())))

cm = confusion_matrix(y_true, y_pred)
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=list(test_gen.class_indices.keys()), yticklabels=list(test_gen.class_indices.keys()))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 7. Ethical reflection

- Data bias: dataset may over/under-represent age groups, geographies, or imaging equipment.
- Fairness: ensure model is validated across different sub-populations before deployment.
- Privacy: chest X-rays are medical data; follow local regulations and anonymize patient metadata.
- Limitations: model should assist clinicians, not replace them. Provide uncertainty estimates and human-in-the-loop workflows.

## 8. Stretch goals & next steps

- Use transfer learning (EfficientNet/BERT for image-text multimodal).
- Deploy as Streamlit/Flask app.
- Integrate real-time hospital data and monitor model drift.